# TypeNext · a small prose FIM training experiment

This notebook fine-tunes **SmolLM2-135M** on a prefix/suffix/middle objective using two pinned public-domain novels. It is a runnable recipe, **not a trained checkpoint or a quality claim**. No notebook has been trained by the maintainers for this change.

In Colab choose **Runtime → Change runtime type → T4 GPU**, then run cells in order. The default run is bounded to 100 optimizer steps and short 384-token sequences. Public model/data downloads occur in your Colab runtime; no TypeNext notes, API keys or private references are read.

The shipped optional model in TypeNext is the original 135M instruction model, not the result of this experiment. Do not replace it with a fine-tune until held-out evaluation, source licensing and tokenizer/export compatibility have been checked.


## Sources and scope

- Base model: [HuggingFaceTB/SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), Apache 2.0, immutable revision 93efa2f097d58c2a74874c7e644dbc9b0cee75a2.
- Train: Jane Austen’s [Pride and Prejudice](https://www.gutenberg.org/ebooks/1342). Validation: Lewis Carroll’s [Alice’s Adventures in Wonderland](https://www.gutenberg.org/ebooks/11). Gutenberg identifies these editions as public domain in the US; check your jurisdiction before redistribution. Their body text is used without the Gutenberg header/license appendix, with provenance retained below. Exact HTTP bytes are SHA-256 pinned; changed bytes stop the run.
- [FIM objective paper](https://arxiv.org/abs/2207.14255): prefix + suffix + middle permutation. This recipe adds explicit special tokens and trains their embeddings with the whole small model. It masks all prompt/suffix labels; only the missing middle and its end marker are supervised.

These two nineteenth-century novels are a **small biased demonstration corpus**. They do not cover modern nonfiction, multilingual writing, factual grounding or individual writing style. A larger consented/licensed corpus and manual evaluation are needed for a useful released writing model. Train/validation split is by entire book **before** windows, so adjacent passages do not cross the split.


In [ ]:
%pip -q install "transformers==4.57.6" "accelerate==1.12.0" "safetensors==0.6.2"


In [ ]:
import json, math, os, random, time
from pathlib import Path
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
set_seed(7)
assert torch.cuda.is_available(), "Select a GPU runtime; this notebook intentionally does not launch a slow CPU training run."
print({"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)})
MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"
OUTPUT = Path("/content/typenext-fim-135m")
MAX_STEPS = 100
MAX_TOKENS = 384
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
"""Small public-data FIM preparation helpers; no model framework or private notes."""
from __future__ import annotations

import hashlib
import random
import re
import urllib.request

BOOKS = {
    "train": {
        "title": "Pride and Prejudice",
        "author": "Jane Austen",
        "url": "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
        "sha256": "3f6bb9d6f78e0293b56acd4714dd68cb7d6d1d293402031ce9d5a216bcaf9d75",
        "bytes": 772386,
    },
    "validation": {
        "title": "Alice's Adventures in Wonderland",
        "author": "Lewis Carroll",
        "url": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
        "sha256": "01b38ea4c710a84bc18d0bd41271a5a1a92b94e97b2812f4dece97d4a694725e",
        "bytes": 174311,
    },
}
FIM_TOKENS = ["<|fim_prefix|>", "<|fim_suffix|>", "<|fim_middle|>", "<|fim_end|>"]


def verified_body(raw: bytes, manifest: dict) -> str:
    """Verify source bytes before stripping Gutenberg headers and license appendix."""
    if len(raw) != manifest["bytes"] or hashlib.sha256(raw).hexdigest() != manifest["sha256"]:
        raise ValueError("Public dataset bytes changed. Review the source before updating its pin.")
    text = raw.decode("utf-8-sig").replace("\r\n", "\n")
    start = re.search(r"^\*\*\* START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK .*?\*\*\*\s*$", text, re.M)
    end = re.search(r"^\*\*\* END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK .*?\*\*\*\s*$", text, re.M)
    if not start or not end or start.end() >= end.start():
        raise ValueError("Missing Gutenberg source boundaries; refusing to train on header/license text.")
    return text[start.end():end.start()].strip()


def download_book(manifest: dict) -> str:
    request = urllib.request.Request(manifest["url"], headers={"User-Agent": "TypeNext-public-FIM-recipe/0.1"})
    with urllib.request.urlopen(request, timeout=45) as response:
        raw = response.read(manifest["bytes"] + 1)
    return verified_body(raw, manifest)


def make_examples(text: str, limit: int = 1024, seed: int = 7, window_words: int = 110) -> list[dict[str, str]]:
    """Deterministic non-overlapping windows; no train/validation mixing."""
    if len(text) > 2_000_000 or not 1 <= limit <= 4096 or not 32 <= window_words <= 180:
        raise ValueError("Dataset preparation limits exceeded.")
    rng = random.Random(seed)
    examples: list[dict[str, str]] = []
    paragraphs = re.split(r"\n\s*\n", text)
    rng.shuffle(paragraphs)
    for paragraph in paragraphs:
        words = list(re.finditer(r"\S+", paragraph))
        for offset in range(0, len(words), window_words):
            window = words[offset:offset + window_words]
            if len(window) < 32:
                continue
            original = paragraph[window[0].start():window[-1].end()]
            local_words = list(re.finditer(r"\S+", original))
            left = rng.randint(8, len(local_words) - 20)
            middle_words = rng.randint(2, min(12, len(local_words) - left - 8))
            begin = local_words[left].start()
            end = local_words[left + middle_words].start()
            example = {"prefix": original[:begin], "middle": original[begin:end], "suffix": original[end:], "original": original}
            assert example["prefix"] + example["middle"] + example["suffix"] == original
            examples.append(example)
            if len(examples) >= limit:
                return examples
    return examples


def fim_prompt_ids(prefix: str, suffix: str, tokenizer) -> list[int]:
    """Identical BOS/PSM prompt ordering for training and later inference."""
    prompt = FIM_TOKENS[0] + prefix + FIM_TOKENS[1] + suffix + FIM_TOKENS[2]
    prefix_ids = tokenizer.encode(prompt, add_special_tokens=False)
    bos = [tokenizer.bos_token_id] if tokenizer.bos_token_id is not None else []
    return bos + prefix_ids


def encode_fim(example: dict[str, str], tokenizer, max_tokens: int = 384) -> dict | None:
    """PSM FIM layout. Only missing text and the end marker contribute to loss."""
    prefix_ids = fim_prompt_ids(example["prefix"], example["suffix"], tokenizer)
    target_ids = tokenizer.encode(example["middle"], add_special_tokens=False) + [tokenizer.convert_tokens_to_ids(FIM_TOKENS[3])]
    input_ids = prefix_ids + target_ids
    if len(input_ids) > max_tokens or len(target_ids) > 40:
        return None
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": [-100] * len(prefix_ids) + target_ids}



In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=torch.float32, attn_implementation="eager", trust_remote_code=False)
added = tokenizer.add_special_tokens({"additional_special_tokens": FIM_TOKENS})
model.resize_token_embeddings(len(tokenizer))
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids(FIM_TOKENS[3])]
model.config.use_cache = False
model.gradient_checkpointing_enable()
print({"parameters": model.num_parameters(), "added_fim_tokens": added, "fim_ids": {token: tokenizer.convert_tokens_to_ids(token) for token in FIM_TOKENS}})


In [ ]:
books = {split: download_book(manifest) for split, manifest in BOOKS.items()}
examples = {split: make_examples(body, limit=1024 if split == "train" else 96, seed=7) for split, body in books.items()}
rows = {split: [row for example in records if (row := encode_fim(example, tokenizer, MAX_TOKENS)) is not None] for split, records in examples.items()}
assert len(rows["train"]) >=100 and len(rows["validation"]) >=16, "Too few bounded FIM examples; inspect source preparation."
# Retain ordinary causal writing in one fifth of additional examples.
for example in examples["train"][::5]:
    ids = tokenizer.encode(example["original"], add_special_tokens=True)[:MAX_TOKENS]
    rows["train"].append({"input_ids": ids, "attention_mask": [1] * len(ids), "labels": ids[:]})
random.Random(7).shuffle(rows["train"])
provenance = {"model": MODEL_ID, "revision": REVISION, "model_license": "Apache-2.0", "dataset": BOOKS, "split": "entire books before windows", "fim_tokens": FIM_TOKENS, "trained": False, "steps": MAX_STEPS, "max_tokens": MAX_TOKENS}
(OUTPUT / "training-provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print({split: len(records) for split, records in rows.items()})


In [ ]:
class PreparedDataset(torch.utils.data.Dataset):
    def __init__(self, records): self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, index): return self.records[index]

def collate(batch):
    length = max(len(item["input_ids"]) for item in batch)
    return {
        "input_ids": torch.tensor([item["input_ids"] + [tokenizer.pad_token_id] * (length - len(item["input_ids"])) for item in batch]),
        "attention_mask": torch.tensor([item["attention_mask"] + [0] * (length - len(item["input_ids"])) for item in batch]),
        "labels": torch.tensor([item["labels"] + [-100] * (length - len(item["input_ids"])) for item in batch]),
    }

arguments = TrainingArguments(
    output_dir=str(OUTPUT / "checkpoints"), max_steps=MAX_STEPS,
    per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=3e-5, warmup_steps=10, weight_decay=0.01,
    fp16=True, gradient_checkpointing=True, logging_steps=10,
    eval_strategy="steps", eval_steps=50, save_strategy="no",
    report_to="none", seed=7, data_seed=7, dataloader_num_workers=0,
)
trainer = Trainer(model=model, args=arguments, train_dataset=PreparedDataset(rows["train"]), eval_dataset=PreparedDataset(rows["validation"]), data_collator=collate, processing_class=tokenizer)
baseline = trainer.evaluate()
print({"untrained_fim_validation_loss": baseline["eval_loss"]})


## Train and record evidence

The baseline above contains newly initialized FIM tokens and is not a production comparison with the app’s instruction model. Validation loss alone is not writing quality. Interrupting this cell leaves `trained:false` provenance and must not be described as a completed fine-tune.


In [ ]:
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
training_result = trainer.train()
validation = trainer.evaluate()
trainer.save_model(str(OUTPUT))
tokenizer.save_pretrained(str(OUTPUT))
model.config.use_cache = True
model.config.save_pretrained(str(OUTPUT))
provenance.update({"trained": True, "training_seconds": time.perf_counter() - started, "peak_cuda_bytes": torch.cuda.max_memory_allocated(), "train_metrics": training_result.metrics, "validation_metrics": validation, "baseline_validation_loss": baseline["eval_loss"], "python_dependencies": {"transformers": "4.57.6", "accelerate": "1.12.0"}})
(OUTPUT / "training-provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print({"training_seconds": provenance["training_seconds"], "peak_cuda_GiB": provenance["peak_cuda_bytes"] /2**30, "before_loss": baseline["eval_loss"], "after_loss": validation["eval_loss"]})


In [ ]:
model.eval()
model.config.use_cache = True
fim_end = tokenizer.convert_tokens_to_ids(FIM_TOKENS[3])

@torch.inference_mode()
def infill(prefix, suffix, max_new_tokens=32):
    ids = fim_prompt_ids(prefix, suffix, tokenizer)
    encoded = {"input_ids": torch.tensor([ids], device=model.device), "attention_mask": torch.ones((1,len(ids)), dtype=torch.long, device=model.device)}
    assert encoded["input_ids"].shape[-1] <=MAX_TOKENS, "Shorten context; do not silently drop the cursor."
    started = time.perf_counter()
    output = model.generate(**encoded, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=fim_end, pad_token_id=tokenizer.pad_token_id)
    new_ids = output[0, encoded["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True), time.perf_counter() - started

checks = []
for example in examples["validation"][:8]:
    insertion, seconds = infill(example["prefix"], example["suffix"])
    checks.append({"prefix": example["prefix"], "suffix": example["suffix"], "expected_public_middle": example["middle"], "generated": insertion, "seconds": seconds, "exact_middle": insertion ==example["middle"], "suffix_echo": example["suffix"][:32].strip() in insertion})
(OUTPUT / "heldout-infill.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")
# Inspect these reconstructions, including suffix boundaries, before treating it as useful.
for item in checks[:3]:
    print({"reconstructed": item["prefix"] + item["generated"] +item["suffix"], "exact_middle": item["exact_middle"], "suffix_echo": item["suffix_echo"]})


## Evaluate the actual writing task

Also test modern synthetic prose: title-only starts, the writer’s objective, existing lowercase suffixes, blank lines, markdown headings, no-reference notes and noisy reference excerpts. Score grammatical compatibility, meaning preservation, suffix duplication and whether the result makes unsupported factual claims. This small corpus has no objective/RAG conditioning examples, so those capabilities must not be inferred from lower loss. Keep automatic suggestions short and retain the deterministic offline fallback.

A released FIM model needs the same tokenizer markers and prompt order at inference. The app’s shipped instruction worker does not automatically discover, download or accept this custom checkpoint. The export below is a developer artifact that needs its own pinned manifest and dedicated FIM formatter; it is not a drop-in change of model ID in the existing chat worker.


## Optional ONNX export and quantization

This can take additional CPU RAM/time. Run it after reviewing evaluation. [Optimum export docs](https://huggingface.co/docs/optimum-onnx/onnx/usage_guides/export_a_model) describe `text-generation-with-past`. An exported file still needs runtime parity tests; quantization can change completions. No unreviewed weights are published or uploaded by this notebook.


In [ ]:
%pip -q install "optimum-onnx==0.1.0" "onnxruntime==1.30.0"


In [ ]:
import gc, shutil, subprocess
# Release training GPU state before a CPU exporter reads the saved checkpoint.
trainer.model = None
del trainer, model
gc.collect()
torch.cuda.empty_cache()
ONNX = Path("/content/typenext-fim-135m-onnx")
subprocess.run(["optimum-cli", "export", "onnx", "--model", str(OUTPUT), "--task", "text-generation-with-past", "--opset", "17", "--device", "cpu", str(ONNX)], check=True)
from onnxruntime.quantization import QuantType, quantize_dynamic
for exported in ONNX.glob("*.onnx"):
    quantized = exported.with_name(exported.stem +"_quantized.onnx")
    quantize_dynamic(str(exported), str(quantized), weight_type=QuantType.QInt8, per_channel=True, op_types_to_quantize=["MatMul"])
# Preserve provenance and special tokens with every export.
shutil.copy2(OUTPUT / "training-provenance.json", ONNX / "training-provenance.json")
shutil.copy2(OUTPUT / "heldout-infill.json", ONNX / "heldout-infill.json")
print({"exported_files": [file.name for file in ONNX.iterdir()]})


In [ ]:
import hashlib
manifest = {file.name: {"bytes": file.stat().st_size, "sha256": hashlib.file_digest(file.open("rb"), "sha256").hexdigest()} for file in ONNX.iterdir() if file.is_file()}
(ONNX / "export-manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive = shutil.make_archive("/content/typenext-fim-135m-onnx", "zip", ONNX)
from google.colab import files
files.download(archive)


## What has and has not been verified

The repository validates notebook JSON/Python syntax, source-byte pins, exact prefix/middle/suffix reconstruction, middle-only loss masks and input bounds. **The training, GPU evaluation, ONNX export and quantized runtime parity cells have not been executed here**: the available Colab browser could not initialize its kernel. Record your completed run’s metrics/provenance instead of treating the recipe as evidence of a trained or improved model.

Never upload your private notes, API-key file, personal PDFs or reference library to this notebook unless you separately choose to process them in Google Colab. The default sources are only the two public novels above.
